# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI


In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [3]:
# set up environment
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key and openai_api_key.startswith('sk-proj-') and len(openai_api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openai = OpenAI()

API key looks good so far


In [4]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [5]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [6]:
system_prompt = """ 
You are an expert AI Engineering tutor and mentor.

Your goal is to answer questions and teach AI Engineering in a structured, practical, and technically rigorous way, 
helping the learner progress from fundamentals to production-level AI systems.

"""

user_prompt = "Please answer the following question: " 

In [7]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt + question}
]

In [8]:
# Get gpt-4o-mini to answer, with streaming
def stream_answer(question, model=MODEL_GPT):
    stream = openai.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=display_handle.display_id)
        
        

In [9]:
stream_answer(question)

This code snippet uses Python's `yield` mechanism along with a set comprehension. Let's break it down to understand what it does step by step:

### Components of the Code

1. **`yield from`**: This is a Python statement that is used in generator functions. It allows you to delegate part of the generator's operations to another generator. Instead of returning a value, it can yield multiple values one by one from another iterable.

2. **Set Comprehension (`{ ... for ... in ...}`)**: Inside the `yield from`, we have a set comprehension. This creates a set (a collection of unique elements) from the data processed. The set comprehension here iterates over each `book` in the `books` iterable.

3. **`book.get("author")`**: This attempts to retrieve the value associated with the key `"author"` from each `book`. If the key `"author"` is not present in the `book` dictionary, `book.get("author")` will return `None`.

4. **`if book.get("author")`**: This condition ensures that only `book` entries that have a non-`None` (or non-falsy) value for `"author"` are included in the set comprehension. If a book has no author (i.e., the value is `None` or evaluates to `False`), it will be omitted.

### What Does the Code Do?

Putting it all together, this code:
- Iterates over a collection called `books`, where each `book` is expected to be a dictionary containing book attributes (one of which is likely `"author"`).
- It fetches the authors of the books, ignoring any that do not have an author listed.
- It creates a set of authors, ensuring that each author appears only once (since sets do not allow duplicates).
- It then yields each unique author from this set one by one.

### Why Use This Code?

1. **Iterating Efficiently**: Using `yield from` allows for an efficient way to provide results without having to construct the entire result set in memory upfront, which is useful for large datasets.

2. **Avoiding Duplicates**: By using a set comprehension, it automatically eliminates duplicate author entries.

3. **Readability**: The use of comprehensions (set comprehension in this case) and `yield from` makes the code more concise and often easier to read.

### Example

Suppose you have the following `books` list:

```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": None},  # No author
    {"title": "Book 4", "author": "Author A"}  # Duplicate author
]
```

The output of the code when run would yield:

```
Author A
Author B
```

The duplicate `"Author A"` is represented only once because of the set's behavior.

In [10]:
import requests

requests.get("http://localhost:11434").content

b'Ollama is running'

In [ ]:
# Get Llama 3.2 to answer
response = ollama.chat.completions.create(
    model=MODEL_LLAMA,   
    messages=messages
)

response.choices[0].message.content